In [ ]:
import torch
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from make_dataset import MultiMatchSoccerDataset
from models.encoder import GraphAutoencoder
from tqdm import tqdm
import os

from make_dataset import MultiMatchSoccerDataset, organize_and_process
from utils.utils import set_evertyhing, worker_init_fn, generator
from utils.data_utils import split_dataset_indices, custom_collate_fn

raw_data_path = "idsse-data"
data_save_path = "match_data"
batch_size = 64
num_workers = 8
epochs = 100
learning_rate = 1e-4
SEED = 42

set_evertyhing(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 1. Load dataset
print("---Data Loading---")
if not os.path.exists(data_save_path) or len(os.listdir(data_save_path)) == 0:
    organize_and_process(raw_data_path, data_save_path)
else:
    print("Skip organize_and_process")

dataset = MultiMatchSoccerDataset(data_root=data_save_path, use_condition_graph=True)
train_idx, _, _ = split_dataset_indices(dataset, val_ratio=1/6, test_ratio=1/6, random_seed=SEED)

train_dataloader = DataLoader(
    Subset(dataset, train_idx),
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers,
    pin_memory=True,
    persistent_workers=False,
    collate_fn=custom_collate_fn,
    worker_init_fn=worker_init_fn,
    generator=generator(SEED)
)

# 2. Input feature dimension extraction
sample = dataset[0]
first_frame = sample["condition_graph_seq"][0]
in_dim_dict = {
    node_type: first_frame[node_type].x.shape[-1]
    for node_type in first_frame.node_types
}


---Data Loading---
Skip organize_and_process


Check:   0%|          | 0/66 [00:00<?, ?it/s]

Batch keys: dict_keys(['match_id', 'condition', 'other', 'target', 'condition_columns', 'other_columns', 'target_columns', 'condition_frames', 'target_frames', 'pitch_scale', 'condition_graph_seq'])


Check:   0%|          | 0/66 [03:02<?, ?it/s]


AttributeError: 'list' object has no attribute 'to'

In [3]:

# 3. 모델 초기화
model = GraphAutoencoder(
    in_dim_dict=in_dim_dict,
    hidden_dim=64,
    temporal_hidden_dim=64,
    out_dim=128  # 최종 H 크기
).to(device)

optimizer = optim.Adam(model.parameters(), lr=1e-3)
num_epochs = 10

for batch in tqdm(train_dataloader, desc="Check"):
    print("Batch keys:", batch.keys())

    condition_seq = [graph.to(device) for graph in batch["condition_graph_seq"]]
    print("Type of condition_seq:", type(condition_seq))
    print("Length of condition_seq (frames):", len(condition_seq))

    first_frame = condition_seq[0]
    print("First frame type:", type(first_frame))

    print(" - Node types:", first_frame.node_types)
    print(" - Edge types:", first_frame.edge_types)

    for node_type in first_frame.node_types:
        print(f"   {node_type} - x :", first_frame[node_type].x)
        print(first_frame[node_type].x.device)

    break

Check:   0%|          | 0/66 [00:00<?, ?it/s]

Batch keys: dict_keys(['match_id', 'condition', 'other', 'target', 'condition_columns', 'other_columns', 'target_columns', 'condition_frames', 'target_frames', 'pitch_scale', 'condition_graph_seq'])
Type of condition_seq: <class 'list'>
Length of condition_seq (frames): 125
First frame type: <class 'abc.HeteroDataBatch'>
 - Node types: ['Attk', 'Def', 'Ball']
 - Edge types: [('Attk', 'interaction', 'Attk'), ('Attk', 'interaction', 'Def'), ('Def', 'interaction', 'Def'), ('Attk', 'interaction', 'Ball'), ('Def', 'interaction', 'Ball')]
   Attk - x : tensor([[ 2.3143e-01, -8.5618e-01,  5.0000e-01,  ...,  6.0572e+03,
          0.0000e+00,  1.0000e+00],
        [ 7.5429e-02,  4.0412e-01, -1.1429e+00,  ...,  7.0074e+03,
          1.3000e+01,  1.0000e+00],
        [ 2.1524e-01,  7.4235e-01, -9.6429e-01,  ...,  6.6490e+03,
          0.0000e+00,  1.0000e+00],
        ...,
        [ 1.9086e-01,  9.3235e-02, -5.0357e+00,  ...,  2.4299e+03,
          3.0000e+00,  1.0000e+00],
        [ 2.7524e-01, 

Check:   0%|          | 0/66 [03:49<?, ?it/s]
